# single_node_sim quickstart

This notebook demonstrates basic usage of the single_node_sim library for simulating EIP-8070 sparse blobpool behavior.

## 1. Import single_node_sim

In [ ]:
from single_node_sim import run
from single_node_sim.events import TxAnnouncement, BlockIncluded, CellsReceived
from single_node_sim.params import HeuristicParams, EvictionPolicy, PRESETS
from single_node_sim.metrics import TxState
from sparse_blobpool.protocol.constants import ALL_ONES

import matplotlib.pyplot as plt

## 2. Basic usage with preset

The simplest way to run a simulation is using the default preset and a list of events.

In [ ]:
events = [
    TxAnnouncement(
        timestamp=0.0,
        tx_hash="0xabc001",
        sender="0xsender1",
        nonce=0,
        gas_fee_cap=1_000_000_000,
        gas_tip_cap=100_000_000,
        tx_size=131_072,
        blob_count=1,
    ),
    TxAnnouncement(
        timestamp=1.0,
        tx_hash="0xabc002",
        sender="0xsender2",
        nonce=0,
        gas_fee_cap=2_000_000_000,
        gas_tip_cap=200_000_000,
        tx_size=262_144,
        blob_count=2,
    ),
]

result = run(events=events, preset="default")
summary = result.metrics.summary()

print(f"Total announcements: {summary.total_announcements}")
print(f"Total completions: {summary.total_completions}")
print(f"Total evictions: {summary.total_evictions}")
print(f"Average completion time: {summary.avg_completion_time:.2f}s")

## 3. Custom parameters example

You can create custom HeuristicParams to test specific scenarios.

In [ ]:
custom_params = HeuristicParams(
    max_pool_bytes=512 * 1024 * 1024,
    max_txs_per_sender=8,
    eviction_policy=EvictionPolicy.HYBRID,
    age_weight=0.3,
    provider_probability=0.20,
    custody_columns=16,
    tx_ttl=120.0,
    seed=42,
)

result_custom = run(events=events, params=custom_params)
summary_custom = result_custom.metrics.summary()

print(f"With custom params:")
print(f"  Total announcements: {summary_custom.total_announcements}")
print(f"  Peak pool size: {summary_custom.peak_pool_size} bytes")
print(f"  Peak tx count: {summary_custom.peak_tx_count}")

## 4. Inspecting SimulationResult

The SimulationResult contains detailed information about the simulation.

In [ ]:
tx_records = result.metrics.get_tx_records()
snapshots = result.metrics.get_snapshots()
debug_log = result.metrics.get_debug_log()
summary = result.metrics.summary()

print("Available result components:")
print(f"  - node: {type(result.node).__name__}")
print(f"  - metrics: {type(result.metrics).__name__}")
print(f"  - final_time: {result.final_time:.2f}s")
print(f"  - tx_records: dict with {len(tx_records)} transactions")
print(f"  - snapshots: list with {len(snapshots)} snapshots")
print(f"  - debug_log: list with {len(debug_log)} entries")

print("\nSummary details:")
print(f"  total_announcements: {summary.total_announcements}")
print(f"  total_completions: {summary.total_completions}")
print(f"  total_evictions: {summary.total_evictions}")
print(f"  avg_completion_time: {summary.avg_completion_time:.3f}s")
print(f"  peak_pool_size: {summary.peak_pool_size:,} bytes")
print(f"  peak_tx_count: {summary.peak_tx_count}")

## 5. Plotting pool size over time with matplotlib

Snapshots capture pool state at each event.

In [ ]:
many_events = [
    TxAnnouncement(
        timestamp=float(i) * 0.5,
        tx_hash=f"0x{i:08x}",
        sender=f"0xsender{i % 10}",
        nonce=i // 10,
        gas_fee_cap=1_000_000_000 + i * 10_000_000,
        gas_tip_cap=100_000_000 + i * 1_000_000,
        tx_size=131_072,
        blob_count=1,
    )
    for i in range(50)
]

result_many = run(events=many_events)
snapshots = result_many.metrics.get_snapshots()

timestamps = [snap.timestamp for snap in snapshots]
sizes = [snap.size_bytes / (1024 * 1024) for snap in snapshots]  # Convert to MB
tx_counts = [snap.tx_count for snap in snapshots]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ax1.plot(timestamps, sizes, 'b-', linewidth=1.5)
ax1.set_ylabel('Pool size (MB)')
ax1.set_title('Blobpool state over time')
ax1.grid(True, alpha=0.3)

ax2.plot(timestamps, tx_counts, 'g-', linewidth=1.5)
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Transaction count')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Inspecting per-tx records

Each transaction has a detailed record of its lifecycle.

In [ ]:
from single_node_sim.metrics import Role

records = result_many.metrics.get_tx_records()

print(f"Total transactions tracked: {len(records)}\n")

sample_hashes = list(records.keys())[:5]
for tx_hash in sample_hashes:
    record = records[tx_hash]
    print(f"Transaction: {tx_hash}")
    print(f"  Role: {record.role.name}")
    print(f"  Announced at: {record.announced_at:.2f}s")
    if record.completed_at:
        print(f"  Completed at: {record.completed_at:.2f}s")
        print(f"  Completion time: {record.completed_at - record.announced_at:.2f}s")
    if record.evicted_at:
        print(f"  Evicted at: {record.evicted_at:.2f}s")
        print(f"  Eviction reason: {record.eviction_reason}")
    print(f"  State history: {[(t, s.name) for t, s in record.state_history]}")
    print()

provider_count = sum(1 for r in records.values() if r.role == Role.PROVIDER)
sampler_count = sum(1 for r in records.values() if r.role == Role.SAMPLER)
print(f"Role distribution:")
print(f"  Providers: {provider_count} ({100*provider_count/len(records):.1f}%)")
print(f"  Samplers: {sampler_count} ({100*sampler_count/len(records):.1f}%)")